# 02 — Feature Engineering

Demonstra o pipeline completo de engenharia de features:

1. WoE encoding via `src.features.build_features`
2. Coleta de séries macro BR via `src.features.macro_features`
3. Análise de correlação e multicolinearidade
4. Preparação final para modelagem

> **Pré-requisito:** `make train` (gera `data/interim/german_credit.parquet` e `data/processed/`)

In [ ]:
from __future__ import annotations
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")

ROOT   = Path().resolve().parent
PARAMS = yaml.safe_load((ROOT / "params.yaml").read_text())

COR_BOM = "#639922"
COR_MAU = "#E24B4A"

df_interim = pd.read_parquet(ROOT / PARAMS["paths"]["interim_parquet"])
df_proc    = pd.read_parquet(ROOT / PARAMS["paths"]["processed_parquet"])
iv_summary = pd.read_csv(ROOT / "data/processed/iv_summary.csv")
woe_maps   = joblib.load(ROOT / PARAMS["paths"]["woe_value_maps"])
feature_names = json.loads((ROOT / "data/processed/feature_columns.json").read_text())

print(f"Interim : {df_interim.shape} | Processed: {df_proc.shape}")
print(f"Features: {len(feature_names)}")
print(f"IV summary: {len(iv_summary)} features")

## 1. WoE Encoding — Transformação das Features

O **Weight of Evidence (WoE)** transforma cada valor de uma feature em log-odds empírico:
```
WoE_i = ln( %Bons_i / %Maus_i )
```
- **WoE > 0:** bin com mais bons do que maus → indicativo positivo
- **WoE < 0:** bin com mais maus → indicativo negativo
- WoE lineariza a relação com o log-odds de inadimplência, ideal para Regressão Logística

In [ ]:
# Visualiza WoE maps das top-6 features por IV
top_features = iv_summary.sort_values("iv", ascending=False).head(6)["feature"].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("WoE por Bin — Top 6 Features (IV)", fontsize=13, fontweight="bold")

for ax, feat in zip(axes.flat, top_features):
    if feat not in woe_maps:
        ax.text(0.5, 0.5, f"'{feat}' sem mapa WoE", ha="center"); continue
    wmap = woe_maps[feat]
    keys = [str(k) for k in wmap.keys()]
    vals = list(wmap.values())
    colors = [COR_BOM if v > 0 else COR_MAU for v in vals]
    bars = ax.bar(range(len(keys)), vals, color=colors)
    ax.axhline(0, color="#555", lw=0.8)
    ax.set_xticks(range(len(keys))); ax.set_xticklabels(keys, rotation=45, ha="right", fontsize=8)
    iv_val = iv_summary.loc[iv_summary["feature"] == feat, "iv"].values
    iv_str = f" (IV={iv_val[0]:.3f})" if len(iv_val) else ""
    ax.set_title(f"{feat}{iv_str}", fontsize=9)
    ax.set_ylabel("WoE")

plt.tight_layout()
plt.savefig(ROOT / "reports" / "woe_bars.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. IV Ranking — Seleção de Features

In [ ]:
# Comparação: features por IV com faixas de poder preditivo
iv_sorted = iv_summary.sort_values("iv", ascending=True)

fig, ax = plt.subplots(figsize=(9, max(5, len(iv_sorted) * 0.35)))

def _iv_color(v):
    if v < 0.02:  return "#cccccc"
    if v < 0.10:  return "#f5a623"
    if v < 0.30:  return COR_BOM
    if v <= 0.50: return "#2c7bb6"
    return COR_MAU

colors = [_iv_color(v) for v in iv_sorted["iv"]]
ax.barh(iv_sorted["feature"], iv_sorted["iv"], color=colors)
ax.axvline(0.02, ls=":",  color="#aaa", lw=1, label="IV 0.02 (mínimo)")
ax.axvline(0.10, ls="--", color="#f5a623", lw=1, label="IV 0.10 (médio)")
ax.axvline(0.30, ls="--", color="#2c7bb6", lw=1, label="IV 0.30 (forte)")
ax.set_xlabel("Information Value")
ax.set_title("Ranking IV — Todas as Features")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "iv_ranking_full.png", dpi=150, bbox_inches="tight")
plt.show()

# Resumo por faixa
faixas = pd.cut(iv_summary["iv"],
                bins=[0, 0.02, 0.10, 0.30, 0.50, np.inf],
                labels=["Sem poder", "Fraco", "Médio", "Forte", "Suspeito"])
print("\nDistribuição por poder preditivo (IV):")
print(faixas.value_counts().sort_index().to_string())

## 3. Correlação entre Features WoE

Features altamente correlacionadas introduzem **multicolinearidade** na Regressão Logística, inflando variâncias dos coeficientes. Features com |correlação| > 0.85 são candidatas à remoção.

In [ ]:
X = df_proc[feature_names]
corr = X.corr()

# Heatmap das top-15 features por IV
top15 = iv_summary.sort_values("iv", ascending=False).head(15)["feature"].tolist()
top15 = [f for f in top15 if f in corr.columns]

fig, ax = plt.subplots(figsize=(11, 9))
corr_sub = corr.loc[top15, top15]
im = ax.imshow(corr_sub.values, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_xticks(range(len(top15))); ax.set_xticklabels(top15, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(top15))); ax.set_yticklabels(top15, fontsize=8)
ax.set_title("Matriz de Correlação — Top 15 Features (WoE)")
for i in range(len(top15)):
    for j in range(len(top15)):
        v = corr_sub.values[i, j]
        if abs(v) > 0.70:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(v) > 0.85 else "black")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "correlacao.png", dpi=150, bbox_inches="tight")
plt.show()

# Pares com correlação alta
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = upper.stack()
high_corr = high_corr[high_corr.abs() > 0.70].sort_values(ascending=False)
if len(high_corr):
    print("\n⚠️  Pares com |correlação| > 0.70 (verificar multicolinearidade):")
    for (f1, f2), v in high_corr.items():
        print(f"  {f1} × {f2}: {v:.3f}")
else:
    print("✅ Nenhum par com correlação > 0.70 entre as features WoE.")

## 4. Séries Macroeconômicas BR

Coleta automática via `python-bcb` (BACEN SGS). As séries são salvas em `data/raw/macro_brasil/`.

In [ ]:
macro_path = ROOT / "data" / "raw" / "macro_brasil"
macro_files = list(macro_path.glob("*.parquet")) if macro_path.is_dir() else []

if macro_files:
    df_macro = pd.concat([pd.read_parquet(f) for f in macro_files], axis=1)
    print(f"Séries disponíveis: {df_macro.columns.tolist()}")
    print(f"Período: {df_macro.index.min()} → {df_macro.index.max()}")
    print(df_macro.tail(5))
else:
    print("Séries macro não coletadas ainda. Execute:")
    print("  poetry run python -m src.features.macro_features")
    print()
    print("As séries serão salvas em data/raw/macro_brasil/ como Parquet.")
    print("Requer conexão com a API do BACEN (BCB SGS).")

## 5. Sumário das Features Finais

In [ ]:
# Estatísticas descritivas das features processadas
desc = df_proc[feature_names].describe().T
desc["iv"] = iv_summary.set_index("feature").reindex(feature_names)["iv"]
desc = desc.sort_values("iv", ascending=False)

print(f"Shape final: {df_proc[feature_names].shape}")
print(f"\nTop 10 features (por IV):")
print(desc[["mean", "std", "min", "max", "iv"]].head(10).to_string())

# Valores ausentes após encoding
null_counts = df_proc[feature_names].isnull().sum()
if null_counts.any():
    print("\n⚠️  Features com valores ausentes:")
    print(null_counts[null_counts > 0])
else:
    print("\n✅ Nenhum valor ausente nas features processadas.")